In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType

CATALOG = "alpenia_dev"
SCHEMA = "bronze"

def table_name(name):
    return f"{CATALOG}.{SCHEMA}.{name}"

print("Environnement de transformation prêt.")

In [0]:
# Chargement des tables 

df_contracts = spark.table(table_name("brz_contracts"))
df_clients = spark.table(table_name("brz_clients"))
df_pricing = spark.table(table_name("brz_pricing"))
df_claims = spark.table(table_name("brz_claims"))
df_web = spark.table(table_name("brz_web_events"))
df_expertises = spark.table(table_name("brz_expertises"))

print("Les six tables Bronze sont chargées.")

In [0]:
# vérification des schémas

for name, df in [
    ("contracts", df_contracts),
    ("clients", df_clients),
    ("pricing", df_pricing),
    ("claims", df_claims),
    ("web_events", df_web),
    ("expertises", df_expertises)
]:
    print(f"\n--- {name} ---")
    df.printSchema()

In [0]:
#   start_date est déjà de type DATE donc on le conserve
display(
    df_contracts.select("start_date").limit(10)
)

df_contracts = df_contracts.withColumn(
    "start_date",
    F.to_date("start_date")
)

In [0]:
# Clients

# Aucune transformation spécifique n'est demandée ici. On conserve les types vérifiés.

# Pricing

# Les deux montants restent en DOUBLE, conformément à ton sujet.

In [0]:
# Claims - Ajout d'un délai de règlement
display(
    df_claims
)
# Calcul du délai de règlement en jours (arrondi à 1 décimale)
df_claims = df_claims.withColumn(
    "settlement_days",
    F.round(
        # Conversion des timestamps en secondes (Unix) et calcul de l'écart en jours (86400s/jour)
        (F.col("settled_ts").cast("long") - F.col("declared_ts").cast("long")) / 86400,
        1
    )
)


In [0]:
# web-event :  convertir le timestamp et aplatir la structure quote

df_web = spark.table(table_name("brz_web_events"))



df_web = (
    df_web
       .withColumn(
        "event_ts",
        F.to_timestamp("event_ts", "yyyy-MM-dd'T'HH:mm:ssX")
    )
    .select(
        "action",
        "channel",
        "client_id",
        "event_id",
        "event_ts",
        F.col("quote.product").alias("quote_product"),
        F.col("quote.premium_eur").alias("quote_premium_eur")
    )
)

display(df_web)

In [0]:
# Expertises  - convertir les dates plus deux colonnes calculées

df_expertises = (
    df_expertises
    .withColumn(
        "opened_at",
        F.to_timestamp("opened_at", "yyyy-MM-dd HH:mm:ss")
    )
    .withColumn(
        "closed_at",
        F.to_timestamp("closed_at", "yyyy-MM-dd HH:mm:ss")
    )
    .withColumn(
        "items_count",
        F.when(
            F.col("items_replaced").isNull(),
            F.lit(0)
        ).otherwise(F.size("items_replaced"))
    )
    .withColumn(
        "is_open",
        F.col("closed_at").isNull()
    )
)

In [0]:
# réécriture des six tables Delta

transformed_tables = {
    "brz_contracts": df_contracts,
    "brz_clients": df_clients,
    "brz_pricing": df_pricing,
    "brz_claims": df_claims,
    "brz_web_events": df_web,
    "brz_expertises": df_expertises
}

for name, df in transformed_tables.items():
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table_name(name))
    )

    print(f"Table mise à jour : {name}")

In [0]:
for name in transformed_tables.keys():
    df = spark.table(table_name(name))

    print(f"\n--- {name} ---")
    print(f"Nombre de lignes : {df.count()}")
    df.printSchema()